# 01 · Caso de negocio — TarifaLatam

> **Borrador (2026-10-30), a partir de `reports/00_caso.md` y `especificacion-proyecto.md`.**
> Lo revisa el **rol Negocio** del grupo: cada frase marcada con ✏️ es una propuesta que el grupo confirma o cambia. La ficha (`reports/00_caso.md`) sigue en estado **borrador** hasta que el instructor la apruebe.

Antes de abrir un solo dato, el proyecto necesita **una pregunta de negocio, una métrica que la refleje y un costo de
equivocarse**. Este notebook no lee `data/raw/`: todo lo que dice sale de la ficha. Las cifras de los datos se miden en el
notebook 02.

## 1. La pregunta de negocio

**Pregunta de negocio:** ¿cuánto debería cobrar por noche un inmueble de Airbnb en América Latina, según sus
características, su barrio y su ciudad?

**¿Quién decide y con qué información?**
- El **anfitrión** fija su tarifa por noche. Hoy tiene la descripción de su inmueble (tipo, capacidad, habitaciones,
  baños, barrio) y lo que ve en los anuncios vecinos. El modelo le da un **precio de referencia en dólares**; él conserva
  la decisión final.
- El **inversionista** decide si compra un inmueble para alquilarlo. Con el precio sugerido estima si el inmueble se
  pagaría.

**¿Qué pasa hoy, sin el modelo?** El anfitrión fija la tarifa "a ojo" o copiando a los vecinos, sin saber cuánto pesa
cada característica. El inversionista no tiene una referencia comparable entre ciudades ni monedas.

**¿Qué NO decide el modelo?** No fija el precio automáticamente ni predice cuánto se ocupará el inmueble: sugiere un
precio publicado de referencia.

## 2. Equivocarse no cuesta lo mismo

En un problema de **regresión** no hay clases ni matriz de costos: el error es la distancia entre el precio sugerido y
el precio real, y puede ser hacia abajo (**subestimar**) o hacia arriba (**sobrestimar**).

| Error | Qué le pasa al anfitrión | Qué le pasa al inversionista |
|---|---|---|
| **Subestimar** (sugerir muy bajo) | Llena el calendario pero deja de ganar dinero en **cada** noche reservada | Descarta un inmueble que sí rendía (pierde una oportunidad) |
| **Sobrestimar** (sugerir muy alto) | El inmueble queda vacío y la plataforma lo baja en las búsquedas | **Compra un inmueble que no rinde lo esperado**: es el error más caro |

**Error más caro:** sobrestimar, para el inversionista.

**Error menos caro:** subestimar por poco, para el anfitrión (el calendario se llena y puede subir el precio después).

**Proporción entre los dos y de dónde sale:** ✏️ *propuesta para el grupo.* No tenemos datos de cuánto pierde cada uno,
así que **no inventamos una proporción**. Para el anfitrión tratamos los dos errores como iguales (1 a 1): por eso la
métrica principal es el **MAE**, que pesa igual hacia arriba y hacia abajo. Para el inversionista no cambiamos la
métrica, pero reportamos aparte el **sesgo** (si el modelo tiende a sobrestimar) en cada ciudad. Si el grupo consigue
una cifra real (por ejemplo, la pérdida típica de una noche vacía frente a una noche barata), se puede pasar a una
pérdida asimétrica en la etapa de optimización.

### Un ejemplo para ver la diferencia entre MAE, RMSE y sesgo

**Qué mide cada una.** Para cada inmueble, el **error** es *precio sugerido − precio real*: sale positivo si el modelo
sugirió de más (sobrestimó) y negativo si sugirió de menos (subestimó).

- **MAE** (*error absoluto medio*) = promedio de |sugerido − real|. Es el promedio de los errores **sin signo**: responde
  "¿por cuántos dólares se equivoca, en promedio, el precio sugerido?". Pesa igual un error de 10 USD hacia arriba que
  uno de 10 USD hacia abajo. Mejor cuanto más cerca de 0.
- **RMSE** (*raíz del error cuadrático medio*) = raíz del promedio de (sugerido − real)². También queda en dólares, pero
  **castiga mucho más los errores grandes**: en el MAE, fallar por 100 USD en un inmueble pesa lo mismo que fallar por
  10 USD en diez inmuebles; en el RMSE pesa diez veces más. Nunca es menor que el MAE; si es mucho mayor, hay unos pocos
  errores muy grandes.
- **Sesgo medio** = promedio de (sugerido − real), **con signo**. No mide cuánto se equivoca el modelo sino **hacia
  dónde**: positivo, tiende a sobrestimar (el error caro del inversionista); negativo, tiende a subestimar; cerca de 0,
  los errores hacia arriba y hacia abajo se compensan. Por eso un modelo puede tener sesgo 0 y aun así un MAE grande.
- **MAPE** (*error porcentual absoluto medio*) = promedio de |sugerido − real| ÷ real, en %. Es el error **en proporción
  al precio**: 10 USD son el 17 % de un inmueble de 60 USD, pero menos del 1 % de uno de 1.500 USD. Sirve para comparar
  ciudades con precios muy distintos. Cuidado: no se puede calcular si el precio real es 0 y se infla en los inmuebles
  muy baratos.
- **R²** (*coeficiente de determinación*) = 1 − suma de (sugerido − real)² ÷ suma de (real − precio promedio)². Dice qué
  parte de las diferencias de precio entre inmuebles explica el modelo, comparado con sugerir siempre el **precio
  promedio**: 1 es perfecto, 0 es igual que sugerir siempre el promedio y negativo es peor que eso. No está en dólares,
  así que no dice cuánto se equivoca el modelo, y como usa cuadrados lo mueven mucho los precios extremos, igual que al
  RMSE. Ojo: el promedio general es una referencia más fácil de superar que la línea base de la sección 3. No aparece en
  el ejemplo de abajo.

Los números de la celda siguiente son **inventados**, solo para mostrar cómo se comporta cada métrica: no son anuncios
reales. Hay cinco inmuebles; el último es un precio extremo, como los que el EDA encontrará.

In [1]:
import numpy as np
import pandas as pd

# Ejemplo inventado: precios reales por noche (USD) y lo que sugieren dos modelos imaginarios
real = np.array([60, 80, 100, 120, 1500])
sugerencias = {
    "A: sobrestima un poco": np.array([70, 90, 110, 130, 1510]),
    "B: subestima un poco": np.array([50, 70, 90, 110, 1490]),
    "C: acierta casi todo, falla el extremo": np.array([60, 80, 100, 120, 900]),
}

filas = []
for nombre, pred in sugerencias.items():
    error = pred - real
    filas.append(
        {
            "modelo": nombre,
            "MAE (USD)": np.abs(error).mean(),
            "RMSE (USD)": np.sqrt((error**2).mean()),
            "sesgo medio (USD)": error.mean(),
            "MAPE (%)": (np.abs(error) / real).mean() * 100,
        }
    )
pd.DataFrame(filas).set_index("modelo").round(1)

,MAE (USD),RMSE (USD),sesgo medio (USD),MAPE (%)
modelo,,,,
A: sobrestima un poco,10.0,10.0,10.0,9.6
B: subestima un poco,10.0,10.0,-10.0,9.6
"C: acierta casi todo, falla el extremo",120.0,268.3,-120.0,8.0


**Lo que muestra el ejemplo:**
- A y B tienen el **mismo MAE** (10 USD), pero A sobrestima y B subestima: el **sesgo** los distingue. Para el
  inversionista, A es peor.
- C acierta cuatro de cinco, pero un solo precio extremo dispara su **RMSE** (268 USD) mucho más que su MAE (120 USD).
  Por eso el RMSE no es la métrica principal: unos pocos precios extremos lo dominarían.
- El **MAPE** pone en proporción el error: 10 USD pesan más sobre un inmueble de 60 USD que sobre uno de 1.500 USD.
  Sirve para comparar ciudades con precios muy distintos (Bogotá frente a Río).
- Ojo: C tiene el **MAPE más bajo** (8 %) aunque su MAE sea el más alto, porque falla en el inmueble caro, donde 600 USD
  son "solo" el 40 %. Ninguna métrica sola cuenta toda la historia: por eso se reportan las tres más el sesgo.

## 3. ¿Contra qué nos comparamos? (línea base)

En regresión, la estrategia sin modelo equivalente a "siempre la clase más frecuente" es **"siempre el precio típico"**.
Usamos una versión un poco más justa, la misma que usaría un anfitrión que copia a sus vecinos:

- **Línea base:** la **mediana del precio por ciudad y tipo de habitación** (`room_type`). Por ejemplo, a toda "vivienda
  completa" de Bogotá se le sugiere la mediana de las viviendas completas de Bogotá.
- Se usa la **mediana** y no el promedio porque los precios extremos arrastran el promedio hacia arriba.

**Lo que dice la línea base:** su MAE en USD es el número que el modelo tiene que bajar. **Todavía no se calcula aquí**:
este notebook no abre los datos. Se calcula en el notebook 03, solo con el conjunto de entrenamiento y con validación
por anfitrión, y se anota en `reports/01_eda.md`.

**¿La exactitud (accuracy) sirve como métrica en tu caso?** No aplica: es una métrica de clasificación. En regresión,
"acertar exacto" un precio casi nunca pasa; lo que importa es **cuánto** se equivoca.

## 4. Cómo se medirá el éxito

- **Métrica principal:** **MAE en USD por noche** (menor es mejor). Se lee directo en dinero: "el precio sugerido se
  equivoca en promedio X dólares", que es como lo entiende el anfitrión. Refleja el costo porque pesa igual subestimar
  que sobrestimar (sección 2).
- **Métricas de control:** **MAPE** (compara ciudades con precios distintos), **R²** y el **sesgo medio** por ciudad
  (vigila la sobrestimación, el error caro del inversionista).
- **Qué resultado haría que el negocio diga «sirve»:** ✏️ *propuesta para el grupo:* que el MAE del modelo sea menor que
  el de la línea base **con un intervalo del 95 % que no incluya el cero**, y que ninguna ciudad quede peor que con la
  línea base. Cuánto menor exactamente lo decide el grupo antes de abrir la prueba.
- **El modelo se entrena en logaritmo del precio** (el precio es muy asimétrico) y se evalúa siempre en dólares.

## 5. Lo que podría salir mal

- **Fuga de información:** en Inside Airbnb, las columnas `price_quote_*` son una cotización del mismo precio, y
  `estimated_revenue_l365d` es precio × ocupación: contienen la respuesta y se eliminan. La ocupación y la disponibilidad
  son **consecuencia** del precio (un precio alto deja el calendario libre): se miden con y sin ellas antes de usarlas.
  Las reseñas solo existen para anuncios con historia: el inversionista que evalúa un inmueble nuevo no las tiene.
- **Sesgos o equidad:** las ciudades tienen tamaños muy distintos (Río 48 mil anuncios, Belice 3,7 mil): el modelo puede
  ajustarse mejor a las grandes. Se reporta el error **por ciudad**. La tasa de cambio afecta los precios en dólares;
  por ese riesgo se excluyó Buenos Aires (decisión del 2026-10-31).
- **Datos personales o confidenciales:** los archivos traen nombre, presentación, fotos y URL del anfitrión. Se eliminan
  al cargar; `host_id` solo se usa para partir los datos, nunca como variable.
- **Hasta dónde puede decidir el modelo y dónde debe decidir una persona:** el modelo **sugiere**; el anfitrión fija el
  precio y el inversionista decide la compra con más información (estado del inmueble, costos, regulación local). La app
  debe mostrar un aviso de uso responsable. El modelo predice el precio **publicado** (no el cobrado), en seis ciudades
  (cinco grandes y Belice), con una foto de septiembre de 2026, sin efecto de temporada.

## 6. Cierre

- [x] Pregunta, quién decide y costos escritos. *(La proporción entre errores queda como 1 a 1 a confirmar: sección 2.)*
- [ ] Línea base calculada. *(Se define aquí; su MAE se calcula en el notebook 03 con el entrenamiento.)*
- [x] Métrica principal elegida y justificada.
- [x] Resumen copiado a `reports/00_caso.md`. *(Ya coincide con la ficha.)*
- [ ] Alguien que **no** escribió este notebook sabe explicar por qué sobrestimar le cuesta más al inversionista.
- [ ] **El rol Negocio revisó el borrador** y el instructor aprobó la ficha (`status: aprobada` en `reports/00_caso.md`).